#--- Preliminary EDA: PO Intervention Data ---#

This notebook provides a basic preliminary profile of the two datasets supplied through Infrastructure Victoria:

- `street_segment_qtr_attributes.csv`
- `sites_db.csv`

The purpose is to understand the dataset structure, identify relevant intervention and construction fields, and note initial data-quality issues before the train subgroup combines intervention timing with public transport patronage data.

This is an exploratory Sprint 1 review only. The original datasets are not altered, and no final treatment sites or analysis methods are selected in this notebook.

In [1]:
#--- Import libraries and files ---#

import pandas as pd
from pathlib import Path

data_folder = Path("data/raw")
list(data_folder.glob("*.csv"))

[PosixPath('data/raw/street_segment_qtr_attributes.csv'),
 PosixPath('data/raw/sites_db.csv')]

In [2]:
#--- 1. Load the Datasets ---#
quarterly_path = data_folder / "street_segment_qtr_attributes.csv"
sites_path = data_folder / "sites_db.csv"

quarterly_df = pd.read_csv(quarterly_path)
sites_df = pd.read_csv(sites_path, encoding="cp1252")

print("quarterly dataset shape:", quarterly_df.shape)
print("sites dataset shape:", sites_df.shape)

quarterly dataset shape: (30783, 15)
sites dataset shape: (637, 35)


In [3]:
#--- 2. Inspect the dataset Structure ---#
print("number of unique quarterly periods:",
      quarterly_df["QUARTER"].nunique()
)

print("quaterly dataset columns:")
display(quarterly_df.columns.tolist())

print("\nsites dataset columns:")
display(sites_df.columns.tolist())

number of unique quarterly periods: 105
quaterly dataset columns:


['Unnamed: 0',
 'QUARTER',
 'SiteID',
 'STREET_SEGMENT_ID',
 'Baseline',
 'InterventionType',
 'InterventionSummary',
 'is_under_construction',
 'other_major_construction',
 'StreetInScopeOnStreetParking',
 'StreetInScopeParkingFormat',
 'StreetInScopeOnStreetParkingCap',
 'BufferOnStreetParking',
 'BufferOnStreetParkingCapacity',
 'BufferOnStreetParkingFormat']


sites dataset columns:


['InterventionID',
 'SiteID',
 'StreetSegmentID',
 'SiteType',
 'Baseline',
 'StreetInScope',
 'InterventionSiteID',
 'DateOfIntervention/InitialCapture',
 'DisruptionStartDate',
 'DisruptionEndDate',
 'InterventionType',
 'NoChangeSince',
 'StreetTrees',
 'StreetTreesChange',
 'StreetLighting',
 'StreetLightingChange',
 'StreetFurniture',
 'StreetFurnitureChange',
 'Placemaking',
 'PlacemakingChange',
 'Wayfinding',
 'WayfindingChange',
 'StreetInScopeOnStreetParking',
 'StreetInScopeParkingFormat',
 'StreetInScopeOnStreetParkingCapacity',
 'BufferOnStreetParking',
 'BufferOnStreetParkingFormat',
 'BufferOnStreetParkingCapacity',
 'OffStreetParking',
 'OffStreetParkingCapacity',
 'NumberOfTrafficLanes',
 'PedestrianEstimatesAvgDay',
 'InterventionSummary',
 'InterventionSummaryExpanded',
 'Comment']

In [4]:
#--- 2.1 Quaterly dataset sample ---#
quarterly_df[
    [
        "QUARTER",
        "SiteID",
        "STREET_SEGMENT_ID",
        "Baseline",
        "InterventionType",
        "is_under_construction",
        "other_major_construction",
        "InterventionSummary"
    ]
].head(10)        

,QUARTER,SiteID,STREET_SEGMENT_ID,Baseline,InterventionType,is_under_construction,other_major_construction,InterventionSummary
0,9900Q2,1,12016,NaN,NaN,0.0,0.0,NaN
1,9900Q3,1,12016,NaN,NaN,0.0,0.0,NaN
2,9900Q4,1,12016,NaN,NaN,0.0,0.0,NaN
3,0001Q1,1,12016,NaN,NaN,0.0,0.0,NaN
4,0001Q2,1,12016,NaN,NaN,0.0,0.0,NaN
5,0001Q3,1,12016,NaN,NaN,0.0,0.0,NaN
6,0001Q4,1,12016,NaN,NaN,0.0,0.0,NaN
7,0102Q1,1,12016,NaN,NaN,0.0,0.0,NaN
8,0102Q2,1,12016,NaN,NaN,0.0,0.0,NaN
9,0102Q3,1,12016,NaN,NaN,0.0,0.0,NaN


In [5]:
#--- 2.2 records containing intervention information ---#
quarterly_df[
    quarterly_df['InterventionType'].notna()
    | quarterly_df['InterventionSummary'].notna()
    | (quarterly_df['is_under_construction'] == 1)
] [
    [
        "QUARTER",
        "SiteID",
        "STREET_SEGMENT_ID",
        "Baseline",
        "InterventionType",
        "is_under_construction",
        "other_major_construction",
        "InterventionSummary"
    ]
    ].head(10)      
    

,QUARTER,SiteID,STREET_SEGMENT_ID,Baseline,InterventionType,is_under_construction,other_major_construction,InterventionSummary
84,2021Q2,1,12016,1.0,Pedestrianisation,1.0,1.0,Baseline pedestrianisation
85,2021Q3,1,12016,0.0,Pedestrianisation,1.0,1.0,Pedestrianisation
189,2021Q2,10,7325,1.0,ProtectedBikeLane,1.0,1.0,Painted bike lane
190,2021Q3,10,7325,NaN,NaN,1.0,1.0,NaN
191,2021Q4,10,7325,NaN,NaN,1.0,1.0,NaN
192,2122Q1,10,7325,0.0,ProtectedBikeLane,1.0,1.0,Protected Bike Lane
294,2021Q2,10,7326,1.0,ProtectedBikeLane,1.0,1.0,Painted bike lane
295,2021Q3,10,7326,NaN,NaN,1.0,1.0,NaN
296,2021Q4,10,7326,NaN,NaN,1.0,1.0,NaN
297,2122Q1,10,7326,0.0,ProtectedBikeLane,1.0,1.0,Protected Bike Lane


In [6]:
#--- 2.3 sites dataset sample ---#
sites_df[
    [
        "InterventionID",
        "SiteID",
        "StreetSegmentID",
        "SiteType",
        "Baseline",
        "DateOfIntervention/InitialCapture",
        "DisruptionStartDate",
        "DisruptionEndDate",
        "InterventionType",
        "InterventionSummary"
    ]
].head(10)    

,InterventionID,SiteID,StreetSegmentID,SiteType,Baseline,DateOfIntervention/InitialCapture,DisruptionStartDate,DisruptionEndDate,InterventionType,InterventionSummary
0,control_id_1_11903,control_id_1,11903,Control,Yes,20251015,NaN,NaN,Control,NA Control Site
1,control_id_10_3391,control_id_10,3391,Control,Yes,20251015,NaN,NaN,Control,NA Control Site
2,control_id_101_4175,control_id_101,4175,Control,Yes,20251004,NaN,NaN,Control,NA Control Site
3,control_id_102_3813,control_id_102,3813,Control,Yes,20251015,NaN,NaN,Control,NA Control Site
4,control_id_103_12024,control_id_103,12024,Control,Yes,20251015,NaN,NaN,Control,NA Control site
5,control_id_104_12025,control_id_104,12025,Control,Yes,20251015,20240623.0,20251015.0,Control,NA Control Site
6,control_id_105_4844,control_id_105,4844,Control,Yes,20251015,NaN,NaN,Control,NA Control Site
7,control_id_105_4845,control_id_105,4845,Control,Yes,20251015,NaN,NaN,Control,NA Control Site
8,control_id_106_6079,control_id_106,6079,Control,Yes,20251015,20140314.0,20230706.0,Control,NA Control site
9,control_id_106_6080,control_id_106,6080,Control,Yes,20251015,20220628.0,20251015.0,Control,NA Control site


In [7]:
#--- 2.4 Data Types ---#
print("quarterly dataset data types:")
display(quarterly_df.dtypes)

print("\nsites dataset data types:")
display(sites_df.dtypes)

quarterly dataset data types:


Unnamed: 0                           int64
QUARTER                             object
SiteID                              object
STREET_SEGMENT_ID                    int64
Baseline                           float64
InterventionType                    object
InterventionSummary                 object
is_under_construction              float64
other_major_construction           float64
StreetInScopeOnStreetParking         int64
StreetInScopeParkingFormat          object
StreetInScopeOnStreetParkingCap      int64
BufferOnStreetParking              float64
BufferOnStreetParkingCapacity       object
BufferOnStreetParkingFormat         object
dtype: object


sites dataset data types:


InterventionID                           object
SiteID                                   object
StreetSegmentID                           int64
SiteType                                 object
Baseline                                 object
StreetInScope                            object
InterventionSiteID                      float64
DateOfIntervention/InitialCapture         int64
DisruptionStartDate                     float64
DisruptionEndDate                       float64
InterventionType                         object
NoChangeSince                           float64
StreetTrees                              object
StreetTreesChange                        object
StreetLighting                           object
StreetLightingChange                     object
StreetFurniture                          object
StreetFurnitureChange                    object
Placemaking                              object
PlacemakingChange                        object
Wayfinding                              

In [8]:
#--- 3. Inital data-quality checks ---#
#--- 3.1 Missing Values in key quarterly fields ---#

quarterly_key_columns = [
        "QUARTER",
        "SiteID",
        "STREET_SEGMENT_ID",
        "Baseline",
        "InterventionType",
        "InterventionSummary",
        "is_under_construction",
        "other_major_construction",
    ]

quarterly_missing = pd.DataFrame({
    "Missing count": quarterly_df[quarterly_key_columns].isna().sum(),
    "Missing percentage": (
        quarterly_df[quarterly_key_columns].isna().mean()*100
    ).round(1)
})

quarterly_missing

,Missing count,Missing percentage
QUARTER,0,0.0
SiteID,0,0.0
STREET_SEGMENT_ID,0,0.0
Baseline,30140,97.9
InterventionType,30140,97.9
InterventionSummary,30140,97.9
is_under_construction,2626,8.5
other_major_construction,1575,5.1


In [9]:
#--- 3.2 Missing Values in key site fields ---#
sites_key_columns = [
        "InterventionID",
        "SiteID",
        "StreetSegmentID",
        "SiteType",
        "Baseline",
        "DateOfIntervention/InitialCapture",
        "DisruptionStartDate",
        "DisruptionEndDate",
        "InterventionType",
        "InterventionSummary"
]

sites_missing = pd.DataFrame({
    "Missing count": sites_df[sites_key_columns].isna().sum(),
    "Missing percentage": (
        sites_df[sites_key_columns].isna().mean()*100
    ).round(1)
})

sites_missing

,Missing count,Missing percentage
InterventionID,0,0.0
SiteID,0,0.0
StreetSegmentID,0,0.0
SiteType,0,0.0
Baseline,0,0.0
DateOfIntervention/InitialCapture,0,0.0
DisruptionStartDate,402,63.1
DisruptionEndDate,348,54.6
InterventionType,5,0.8
InterventionSummary,0,0.0


In [10]:
#--- 3.3 duplicate street segment and quarter records ---#
duplicate_quarter_rows = quarterly_df.duplicated(
    subset=["STREET_SEGMENT_ID", "QUARTER"],
    keep=False
)

print("row involved in duplicate segment-quarter combinations:",
      duplicate_quarter_rows.sum())

print("number of duplicate segment quarter combinations:",
      quarterly_df.loc[
            duplicate_quarter_rows,
            ["STREET_SEGMENT_ID", "QUARTER"]
          ].drop_duplicates().shape[0])

row involved in duplicate segment-quarter combinations: 246
number of duplicate segment quarter combinations: 123


In [11]:
#--- 3.4 exact duplicate rows ---#

exact_duplicate_count = quarterly_df.duplicated().sum()
print("number of exact duplicate rows:", exact_duplicate_count)

number of exact duplicate rows: 0


In [12]:
#--- 3.5 inspect repeated segment-quarter records ---#
quarterly_df.loc[
    duplicate_quarter_rows,
    [
        "QUARTER",
        "SiteID",
        "STREET_SEGMENT_ID",
        "Baseline",
        "InterventionType",
        "InterventionSummary",
        "is_under_construction",
        "other_major_construction",
    ]
    ].sort_values(
        ['STREET_SEGMENT_ID', 'QUARTER']
    ).head(20)

,QUARTER,SiteID,STREET_SEGMENT_ID,Baseline,InterventionType,InterventionSummary,is_under_construction,other_major_construction
4609,2122Q2,42,123,1.0,ProtectedBikeLane,No bike lane of any type,1.0,1.0
4610,2122Q2,42,123,0.0,ProtectedBikeLane,Protected bike lane,1.0,1.0
4715,2122Q2,42,124,1.0,ProtectedBikeLane,No bike lane of any type,1.0,1.0
4716,2122Q2,42,124,0.0,ProtectedBikeLane,Protected bike lane (open one direction),1.0,1.0
4821,2122Q2,42,125,1.0,ProtectedBikeLane,No bike lane of any type,1.0,1.0
4822,2122Q2,42,125,0.0,ProtectedBikeLane,Protected bike lane,1.0,1.0
3131,2021Q4,34,3124,1.0,ProtectedBikeLane,Painted bike lane,1.0,1.0
3132,2021Q4,34,3124,0.0,ProtectedBikeLane,Protected bike lane,1.0,1.0
12830,0001Q1,control_id_131,3372,NaN,NaN,NaN,0.0,0.0
28577,0001Q1,48,3372,NaN,NaN,NaN,NaN,NaN


In [13]:
#--- 3.6 test a more detailed record key ---#
detailed_duplicate_rows = quarterly_df.duplicated(
    subset=[
        "SiteID",
        "STREET_SEGMENT_ID",
        "QUARTER",
        "Baseline"
    ],
    keep=False
)

print(
    "rows repeated using SiteID + segment + quarter + baseline:",
    detailed_duplicate_rows.sum()
)

rows repeated using SiteID + segment + quarter + baseline: 14


In [14]:
#--- 3.7 inspect reamining repeated records ---#
quarterly_df.loc[
    detailed_duplicate_rows,
    [
        "QUARTER",
        "SiteID",
        "STREET_SEGMENT_ID",
        "Baseline",
        "InterventionType",
        "InterventionSummary",
        "is_under_construction",
        "other_major_construction",
        "StreetInScopeOnStreetParking",
        "StreetInScopeParkingFormat",        
        "StreetInScopeOnStreetParkingCap"
    ]
].sort_values(
    ["SiteID", "STREET_SEGMENT_ID", "QUARTER", "Baseline"]
)
        

,QUARTER,SiteID,STREET_SEGMENT_ID,Baseline,InterventionType,InterventionSummary,is_under_construction,other_major_construction,StreetInScopeOnStreetParking,StreetInScopeParkingFormat,StreetInScopeOnStreetParkingCap
3664,2223Q3,39,5629,0.0,ProtectedBikeLane,Protected bike lane construction,0.0,1.0,1,Parallel,4
3665,2223Q3,39,5629,0.0,ProtectedBikeLane,Protected bike lane+other major construction,0.0,1.0,1,Parallel,4
3762,2021Q3,39,5637,0.0,ProtectedBikeLane,Post-protected bike lane (bike lane alignment ...,0.0,1.0,1,Parallel,0
3763,2021Q3,39,5637,0.0,ProtectedBikeLane,Protected bike lane construction,0.0,1.0,1,Parallel,0
3863,1920Q2,39,5638,0.0,ProtectedBikeLane,Protected bike lane+other major construction,0.0,1.0,1,Parallel,7
3864,1920Q2,39,5638,0.0,ProtectedBikeLane,Protected bike lane construction,0.0,1.0,1,Parallel,7
3974,2021Q3,39,5639,0.0,ProtectedBikeLane,Protected Bike Lane,1.0,1.0,1,Parallel,0
3975,2021Q3,39,5639,0.0,ProtectedBikeLane,Protected bike lane construction,1.0,1.0,1,Parallel,0
4490,1819Q1,41,12021,0.0,ProtectedBikeLane,Protected bike lane construction + other major...,1.0,1.0,1,Parallel,16
4491,1819Q1,41,12021,0.0,ProtectedBikeLane,Protected bike lane construction,1.0,1.0,1,Parallel,16


In [15]:
#--- 3.8 construction flag values ---#

print("is_under_construction_values:")
display(
    quarterly_df["is_under_construction"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nother_major_construction values:")
display(
    quarterly_df["other_major_construction"]
    .value_counts(dropna=False)
    .sort_index()
)

is_under_construction_values:


is_under_construction
0.0    27968
1.0      189
NaN     2626
Name: count, dtype: int64


other_major_construction values:


other_major_construction
0.0    28607
1.0      601
NaN     1575
Name: count, dtype: int64

In [16]:
#--- 3.9 street segments with construction activity ---#
construction_segment_summary = pd.Series({
    "total unique street segments":
        quarterly_df['STREET_SEGMENT_ID'].nunique(),

    "segments with is_under_construction = 1":
        quarterly_df.loc[
            quarterly_df['is_under_construction'] == 1,
                "STREET_SEGMENT_ID"
            ].nunique(),

    "segments with other_major_construction = 1":
        quarterly_df.loc[
            quarterly_df['other_major_construction'] == 1,
                "STREET_SEGMENT_ID"
                ].nunique()
})

construction_segment_summary

total unique street segments                  292
segments with is_under_construction = 1        69
segments with other_major_construction = 1     99
dtype: int64

In [17]:
#--- 4. compare street segment coverafe across datasets ---#
quarterly_segments = set(quarterly_df["STREET_SEGMENT_ID"].dropna())
sites_segments = set(sites_df["StreetSegmentID"].dropna())

print("segments in both datasets:", len(quarterly_segments & sites_segments))
print("segments only in quarterly dataset:", len(quarterly_segments - sites_segments))
print("segments only in sites dataset:", len(sites_segments - quarterly_segments))

segments in both datasets: 269
segments only in quarterly dataset: 23
segments only in sites dataset: 5


In [18]:
#--- 4.1 unmatched street-segment IDs ---#
print("segments only in quarterly dataset:")
print(sorted(quarterly_segments - sites_segments))

print("\nsegments only in sites dataset:")
print(sorted(sites_segments - quarterly_segments))

segments only in quarterly dataset:
[1563, 1586, 1587, 2559, 3504, 3750, 3751, 3752, 3753, 5753, 6237, 6708, 6709, 7072, 7115, 7336, 7851, 9011, 9649, 9650, 9651, 10396, 10398]

segments only in sites dataset:
[3500, 3794, 3830, 5640, 11482]


In [19]:
#--- 4.2 street segments with no recorded construction flag ---#
segments_without_construction_flag = (
    quarterly_df
    .groupby("STREET_SEGMENT_ID")["is_under_construction"]
    .apply(lambda values: values.isna().all())
)

print("segments where is_under_construction is missing for every quarter:", 
      segments_without_construction_flag.sum()
     )

segments where is_under_construction is missing for every quarter: 24


#--- 5. preliminary findings and limitations ---#

This notebook completed an initial profile of the two intervention datasets provided through the Product Owner. No records were altered, removed or overwritten during the analysis. 

#--- KEY FINDINGS
- the quarterly dataset contains an Unnamed: 0 field, which appears to be an index retained during CSV export and is not required for the current analysis.
- The quarterly dataset contains 30,783 rows, 292 unique street segments and 105 quarterly periods.
- The sites dataset contains 637 rows and 274 unique street segments.
- the main quarterly identifiers (QUARTER, SiteID and STREET_SEGMENT_ID) contain no missing values.
- intervention descriptions are populated only in a small proportion of quarterly records, which appears to reflect the structure of the dataset rather than necessarily indicating missing data.
- the is_under_construction field is missing 8.5% of quarterly rows, while other_major_construction is missing in 5.1%.
- there are 123 repeated street segment and quarter combinations involving 246 rows. However, there are no exact duplicate rows.
- some repeated records represent different baseline and post-intervention sates, different site IDs, or different intervention descriptions. they should therefore not be removed without clarification.
- of the 292 street segments, 69 have at least one quarter flagged as being under intervention construction.
- a further 99 street segments have at least one quarter flagged for other major construction.
- 24 street segments have no recorded is_under_construction value across their entire quarterly history.
- street-segment coverage aligns reasonably well across the two files: 269 segments appear in both, 23 appear only in the quarterly dataset, and 5 appear only in the sites dataset.

#--- LIMITATIONS AND NEXT STEPS
- missing construction values should not automatically be interpreted as zero.
- the repeated records require clarification before aggregation or deduplication.
- the unmatched street-segment IDs should be investigated before joining the datasets.
- the datasets do not contain latitude, longitude, geometry fields. sites_db provides some location context though StreetInScope, but an additional spatial source will still be required for matching interventions to train stations.
- this is a preliminary sprint 1 review only. no final treatment sites, construction windows or before and after analysis methods have been selected.

In [20]:
#--- 4.3 Apply Direction from Stream A Lead 10/08 ---#

import re

#work on a copy to keep original dataframe unchanged
working_df = quarterly_df.copy()

#convert financial year quarter labels into dates to be sorted chronologically

def quarter_start_date(value):
    match = re.fullmatch(r"(\d{2})(\d{2})Q([1-4])", str(value))
    
    if not match:
        return pd.NaT
    
    start_year_short = int(match.group(1))
    quarter = int(match.group(3))
    
    if start_year_short >= 90:
        start_year = 1900 + start_year_short
    else:
        start_year = 2000 + start_year_short
    
    month = {
        1: 7, 
        2: 10,
        3: 1,
        4: 4
    }[quarter]
    
    calendar_year = (
        start_year
        if quarter in [1,2]
        else start_year +1
        )
    
    return pd.Timestamp(
        year=calendar_year,
        month=month,
        day=1
        )

#create sortable quarter date
working_df["quarter_start"] = (
    working_df["QUARTER"]
        .apply(quarter_start_date)
)

#preserve original row orrder where multiple records occur in one quarter
working_df["_original_order"] = range(len(working_df))

working_df = working_df.sort_values(
    [
        "STREET_SEGMENT_ID",
        "quarter_start",
        "_original_order"
    ]
)

#instruction from Team A lead:
# forwardfill, then backward fill, within each street segment
fill_columns = [
    "InterventionType",
    "InterventionSummary"
]

working_df[fill_columns] = (
    working_df
    .groupby("STREET_SEGMENT_ID")[fill_columns]
    .transform(lambda column: column.ffill().bfill())
)

print("missing values after fill:")
display(
    working_df[fill_columns]
    .isna()
    .sum()
)    

missing values after fill:


InterventionType       0
InterventionSummary    0
dtype: int64

In [21]:
#--- 4.4 confirm the segments with missing information ---#
segments_missing_construction = sorted(
    segments_without_construction_flag[
        segments_without_construction_flag
        ].index.tolist() 
)

print("segments only in quarterly dataset:")
print(sorted(quarterly_segments - sites_segments))

print("\nSegments only in sites dataset:")
print(sorted(sites_segments - quarterly_segments))

print("\nsegments with is_under_construction missing for every quarter:")
print(segments_missing_construction)

segments only in quarterly dataset:
[1563, 1586, 1587, 2559, 3504, 3750, 3751, 3752, 3753, 5753, 6237, 6708, 6709, 7072, 7115, 7336, 7851, 9011, 9649, 9650, 9651, 10396, 10398]

Segments only in sites dataset:
[3500, 3794, 3830, 5640, 11482]

segments with is_under_construction missing for every quarter:
[1563, 1586, 1587, 2559, 3503, 3504, 3750, 3751, 3752, 3753, 5753, 6237, 6708, 6709, 7072, 7115, 7336, 7851, 9011, 9649, 9650, 9651, 10396, 10398]


In [22]:
#--- 4.5 check relatinoship between unmatched and missing consturction segments ---#

quarterly_only = quarterly_segments - sites_segments
missing_construction = set(segments_missing_construction)

print(
    "quarterly-only segments also missing all construction values:",
    len(quarterly_only & missing_construction)
)

print(
    "missing construction segments that are NOT quarterly only:",
    sorted(missing_construction - quarterly_only)
)

quarterly-only segments also missing all construction values: 23
missing construction segments that are NOT quarterly only: [3503]


In [23]:
#--- 4.6 Identfy invetention completion quarters ---#
baseline_zero = working_df[
    working_df["Baseline"] == 0
].copy()

print(
    "rows where baseline == 0:",
    len(baseline_zero)
)

print(
    "street segments with at least one baseline == 0:",
    baseline_zero["STREET_SEGMENT_ID"].nunique()
)

display(
    baseline_zero[
        [
            "STREET_SEGMENT_ID",
            "QUARTER",
            "Baseline",
            "InterventionType",
            "InterventionSummary"
        ]
    ].head(20)
)

    

rows where baseline == 0: 350
street segments with at least one baseline == 0: 87


,STREET_SEGMENT_ID,QUARTER,Baseline,InterventionType,InterventionSummary
4610,123,2122Q2,0.0,ProtectedBikeLane,Protected bike lane
4620,123,2324Q4,0.0,ProtectedBikeLane,Post-protected bike lane (no change)
4716,124,2122Q2,0.0,ProtectedBikeLane,Protected bike lane (open one direction)
4726,124,2324Q4,0.0,ProtectedBikeLane,Protected bike lane
4822,125,2122Q2,0.0,ProtectedBikeLane,Protected bike lane
4832,125,2324Q4,0.0,ProtectedBikeLane,Post-protected bike lane (no change)
3447,858,2021Q4,0.0,ProtectedBikeLane,Protected bike lane
3552,859,2021Q4,0.0,ProtectedBikeLane,Protected bike lane
28404,1587,0910Q2,0.0,Regional Reallocation,Pre-reallocation (construction)
28406,1587,0910Q4,0.0,Regional Reallocation,Reallocation


#--- 10/08/2026: Initial PO Internvention EDA ---#
- quarterly dataset contains 30,783 records and 292 unique street segments.
- sites dataset contains 637 records and 274 unique street segments.
- 269 street segments occur in both datasets.
- 23 only occur in the quarterly dataset and 5 only in sites_db.
- 24 street segments have is_under_construction missing for their entire quarterly history.
- repeated segment-quarter records were identfied and retained rather than removed.

#--- 10/08/2026: Stream A / PO Clarification ---#
- confirmed InterventionType and InterventionSummary should be forward filled and then backward filled within each street segment.
- applied the approach to a working copy of the data.
- result: 0 missing InterventionType values and 0 missing InterventionSummary values.
- Confirmed Baseline == 0 should be used to identify construction completion.

#--- 11/08/2026: Construction Missingness Investigation ---#
- 24 segments have is_under_construction missing for their entire history
- 23 of 24 segments are the same segments that occur only in the quarterly dataset.
- sgment 3503 is the only additional segment.
- missing construction values continue to be retained as missing rather than assumed to equal 0.

#--- 11/08/2026: Baseline completion investigation ---#
- 350 records have Baseline == 0.
- these represent 87 unique street segments.
- multiple Baseline == 0 records occur for some segments.
- next check: determine whether first baseline == 0 consistently represents the transition to completed intervention status.

#--- 11/08/2026 ---#
- the earlier team A information referred to missing cycling-infrastructure entires for some segments., subequent team discussoin indicated this may have resulted from a misreading of the Wellington data and is being clarified separarely. this should not currently be treated as the same issues as teh PO intervention dataset gaps identified in this EDA.
- the confirmed PO instruction to forward-fill then backward-fill InterventionType and InterventionSummary remains applicable and has been successfully implemented. missing is_under_construction values have not been filled, as no instruction has been provided. 


#--- Potential next actions ---#
- validate Baseline transition behaviour.
- product segment-level intervention/timing summary
- identify records with strongest timing and location information.
- confirm train subgroup sprint 1 findings and outstanding blockers.
- use subgroup discussion to determine shortlist/matching priorities for sprint 2.

In [24]:
#--- 4.7 Validate intervention completion quarter ---#

# find the first quarter where Baseline == 0 for each street segment
completion_quarters = (
    baseline_zero
    .sort_values(["STREET_SEGMENT_ID", "quarter_start"])
    .groupby("STREET_SEGMENT_ID")
    .first()
    .reset_index()
)

completion_quarters = completion_quarters[
    [
        "STREET_SEGMENT_ID",
        "SiteID",
        "QUARTER",
        "quarter_start",
        "InterventionType",
        "InterventionSummary"
    ]
]

completion_quarters = completion_quarters.rename(
    columns={
        "QUARTER": "completion_quarter",
        "quarter_start": "completion_quarter_start",
    }
)

print(
    "street segments with an identified completion quarter:",
    completion_quarters["STREET_SEGMENT_ID"].nunique()
)

display(completion_quarters.head(20))

street segments with an identified completion quarter: 87


,STREET_SEGMENT_ID,SiteID,completion_quarter,completion_quarter_start,InterventionType,InterventionSummary
0,123,42,2122Q2,2021-10-01,ProtectedBikeLane,Protected bike lane
1,124,42,2122Q2,2021-10-01,ProtectedBikeLane,Protected bike lane (open one direction)
2,125,42,2122Q2,2021-10-01,ProtectedBikeLane,Protected bike lane
3,858,36,2021Q4,2021-04-01,ProtectedBikeLane,Protected bike lane
4,859,36,2021Q4,2021-04-01,ProtectedBikeLane,Protected bike lane
5,1587,46,0910Q2,2009-10-01,Regional Reallocation,Pre-reallocation (construction)
6,1659,30,1011Q4,2011-04-01,ProtectedBikeLane,Pre-protected bike lane (minor change)
7,1660,30,1011Q4,2011-04-01,ProtectedBikeLane,Pre-protected bike lane (minor change)
8,1661,30,1011Q4,2011-04-01,ProtectedBikeLane,Pre-protected bike lane (minor change)
9,1662,30,1011Q4,2011-04-01,ProtectedBikeLane,Pre-protected bike lane (minor change)


Records have been grouped by street segment and identified the earliest quarter where Baseline == 0, following the PO guidance for definding the intervention completion. 

**Inital Finding:**
A candidate intervention-completion quarter was identifed for 87 street segments using the earliest record where 'Baseline == 0', following PO guidance

In [25]:
#--- 4.8 Check baseline segment sequence after candidate completion ---#
# Add the proprosed completion quarter bak onto the quarterly records
baseline_check = working_df.merge(
    completion_quarters[
        ["STREET_SEGMENT_ID","completion_quarter_start"]
    ],
    on="STREET_SEGMENT_ID",
    how="inner"
)

#find the records occuring after the proposed completion quarter
records_after_completion = baseline_check[
    baseline_check["quarter_start"]
    > baseline_check["completion_quarter_start"]
]

#check whether any segments return to Baseline == 1 afterwards
baseline_returns_to_one = records_after_completion[
    records_after_completion["Baseline"] == 1
]

print(
    "Street segments that return to Baseline == 1 after the candidate completion quarter:",
    baseline_returns_to_one["STREET_SEGMENT_ID"].nunique()
)

display(
    baseline_returns_to_one[
        [
            "STREET_SEGMENT_ID",
            "SiteID",
            "QUARTER",
            "Baseline",
            "InterventionType",
            "InterventionSummary"
        ]
    ].head(20)
)

Street segments that return to Baseline == 1 after the candidate completion quarter: 0


,STREET_SEGMENT_ID,SiteID,QUARTER,Baseline,InterventionType,InterventionSummary


**Finding:**
None of the 87 street segments return to 'Baseline == 1' after their candidate completion quarter. This supports using the earliest 'Baseline  == 0' records as the intervention-completion point for the preliminary treatment window analysis..

In [26]:
#--- 4.9 create preliminary intervention summary ---#

intervention_summary = completion_quarters.copy()

print(
    "street segments in preliminary intervention summary:",
    intervention_summary["STREET_SEGMENT_ID"].nunique()
)

display(intervention_summary.head(20))

street segments in preliminary intervention summary: 87


,STREET_SEGMENT_ID,SiteID,completion_quarter,completion_quarter_start,InterventionType,InterventionSummary
0,123,42,2122Q2,2021-10-01,ProtectedBikeLane,Protected bike lane
1,124,42,2122Q2,2021-10-01,ProtectedBikeLane,Protected bike lane (open one direction)
2,125,42,2122Q2,2021-10-01,ProtectedBikeLane,Protected bike lane
3,858,36,2021Q4,2021-04-01,ProtectedBikeLane,Protected bike lane
4,859,36,2021Q4,2021-04-01,ProtectedBikeLane,Protected bike lane
5,1587,46,0910Q2,2009-10-01,Regional Reallocation,Pre-reallocation (construction)
6,1659,30,1011Q4,2011-04-01,ProtectedBikeLane,Pre-protected bike lane (minor change)
7,1660,30,1011Q4,2011-04-01,ProtectedBikeLane,Pre-protected bike lane (minor change)
8,1661,30,1011Q4,2011-04-01,ProtectedBikeLane,Pre-protected bike lane (minor change)
9,1662,30,1011Q4,2011-04-01,ProtectedBikeLane,Pre-protected bike lane (minor change)


### 4.10 check site-record matches for the intervention summary
before adding site-level location and date inforamtion, the number of matching 'sites_db.csv' records is checked for each intervention street segment. this avoids unintentionally creating duplicate rows when the datasets are joined. 

In [27]:
#--- 4.10 check site-record matches before joining ---#

#keep only site records that relate to the 87 intervention segments
matching_sites = sites_df[
    sites_df["StreetSegmentID"].isin(
        intervention_summary["STREET_SEGMENT_ID"]
    )
].copy()

#count how many sites_db records match each street segment
site_match_counts = (
    matching_sites
    .groupby("StreetSegmentID")
    .size()
    .reset_index(name="site_record_count")
)

#identify segments with no matching sites_db record
matched_segment_ids = set(matching_sites["StreetSegmentID"])

no_site_match = intervention_summary[
    ~intervention_summary["STREET_SEGMENT_ID"].isin(matched_segment_ids)
]

#identify segments with more than one matching sites_db record
multiple_site_matches = site_match_counts[
    site_match_counts["site_record_count"] > 1
]

print(
    "intervention segments:",
    intervention_summary["STREET_SEGMENT_ID"].nunique()
)

print(
    "segments with a sites_db match:",
    site_match_counts["StreetSegmentID"].nunique()
)

print(
    "segments with no sites_db match:",
    no_site_match["STREET_SEGMENT_ID"].nunique()
)

print(
    "segments with multiple sites_db match:",
    multiple_site_matches["StreetSegmentID"].nunique()
)

display(multiple_site_matches.head(20))
    

intervention segments: 87
segments with a sites_db match: 76
segments with no sites_db match: 11
segments with multiple sites_db match: 76


,StreetSegmentID,site_record_count
0,123,3
1,124,3
2,125,3
3,858,2
4,859,2
5,1659,12
6,1660,12
7,1661,12
8,1662,12
9,1663,12


**Finding:**
Of the 87 intervention street segments, 76 have a corresponding record in 'sites_db', while 11 do not. All 76 matched street segments are associated with multiple 'sites_db' records, so 'StreetSegmentID' alone is not sufficiently specific for a one-to-one join. The multiple matching records were therefore inspected before any further joining was attempted.

In [28]:
#--- 4.11 inspect multiple sites_db records for selected segments ---#

#choose a few example segments with different numbers of matching records
example_segments = [123, 858, 1659, 3503]

example_site_records = sites_df[
    sites_df["StreetSegmentID"].isin(example_segments)
][
    [
        "InterventionID",
        "SiteID",
        "StreetSegmentID",
        "SiteType",
        "Baseline",
        "StreetInScope",
        "InterventionSiteID",
        "DateOfIntervention/InitialCapture",
        "DisruptionStartDate",
        "DisruptionEndDate",
        "InterventionType",
        "InterventionSummary"
    ]
].sort_values(
    ["StreetSegmentID", "SiteID"]
)

display(example_site_records)

,InterventionID,SiteID,StreetSegmentID,SiteType,Baseline,StreetInScope,InterventionSiteID,DateOfIntervention/InitialCapture,DisruptionStartDate,DisruptionEndDate,InterventionType,InterventionSummary
406,site_id_42_123,site_id_42,123,Intervention,Yes,GHERINGHAP STREET,NaN,20211006,NaN,NaN,ProtectedBikeLane,No bike lane of any type
407,site_id_42_123,site_id_42,123,Intervention,No,GHERINGHAP STREET,NaN,20211122,20211006.0,20211122.0,ProtectedBikeLane,Protected bike lane
408,site_id_42_123,site_id_42,123,Intervention,No,GHERINGHAP STREET,NaN,20240423,20240204.0,20240423.0,ProtectedBikeLane,Post-protected bike lane (no change)
349,site_id_36_858,site_id_36,858,Intervention,Yes,RATHDOWNE STREET,NaN,20210122,NaN,NaN,ProtectedBikeLane,Painted bike lane
350,site_id_36_858,site_id_36,858,Intervention,No,RATHDOWNE STREET,NaN,20210429,20210122.0,20210429.0,ProtectedBikeLane,Protected bike lane
264,site_id_30_1659,site_id_30,1659,Intervention,Yes,abbotsford st,30.0,20091012,NaN,NaN,ProtectedBikeLane,Painted bike lane
265,site_id_30_1659,site_id_30,1659,Intervention,No,abbotsford st,30.0,20110406,20110129.0,20110406.0,ProtectedBikeLane,Pre-protected bike lane (minor change)
266,site_id_30_1659,site_id_30,1659,Intervention,No,abbotsford st,30.0,20130621,NaN,20130621.0,ProtectedBikeLane,Pre-protected bike lane (no change)
267,site_id_30_1659,site_id_30,1659,Intervention,No,abbotsford st,30.0,20130904,NaN,20130904.0,ProtectedBikeLane,Pre-protected bike lane (no change)
268,site_id_30_1659,site_id_30,1659,Intervention,No,abbotsford st,30.0,20181019,20180823.0,20181019.0,ProtectedBikeLane,Pre-protected bike lane (minor change)


**Finding:** A sample of matched  records shows that some street segments have several observations recorded at different points in time. These are ot simple duplicate rows, so they have been retained rather than automatically combined or removed.

In [29]:
#--- 4.12 summarise reocrded construction periods ---#

#keep quarters where the intervention was recorded as under construction
construction_records = working_df[
    working_df["is_under_construction"] == 1
].copy()

#find first and last recorded construction quarter for each segment
construction_periods = (
    construction_records
    .groupby("STREET_SEGMENT_ID")
    .agg(
        construction_start=("quarter_start","min"),
        construction_end=("quarter_start", "max")
    )
    .reset_index()
)

#add construction periods to the preliminary intervention summary
sprint1_intervention_summary = intervention_summary.merge(
    construction_periods,
    on="STREET_SEGMENT_ID",
    how="left"
)

print(
    "Intervention segments:",
    len(sprint1_intervention_summary)
)

print(
    "segments with a recorded cosntruction period:",
    sprint1_intervention_summary["construction_start"].notna().sum()
)

print(
    "segments without a recorded construction period:",
    sprint1_intervention_summary["construction_start"].isna().sum()
)

display(
    sprint1_intervention_summary[
        [
            "STREET_SEGMENT_ID",
            "SiteID",
            "InterventionType",
            "InterventionSummary",
            "completion_quarter",
            "construction_start",
            "construction_end"
        ]
    ].head(20)
)

Intervention segments: 87
segments with a recorded cosntruction period: 69
segments without a recorded construction period: 18


,STREET_SEGMENT_ID,SiteID,InterventionType,InterventionSummary,completion_quarter,construction_start,construction_end
0,123,42,ProtectedBikeLane,Protected bike lane,2122Q2,2021-10-01,2021-10-01
1,124,42,ProtectedBikeLane,Protected bike lane (open one direction),2122Q2,2021-10-01,2024-04-01
2,125,42,ProtectedBikeLane,Protected bike lane,2122Q2,2021-10-01,2021-10-01
3,858,36,ProtectedBikeLane,Protected bike lane,2021Q4,2021-01-01,2021-04-01
4,859,36,ProtectedBikeLane,Protected bike lane,2021Q4,NaT,NaT
5,1587,46,Regional Reallocation,Pre-reallocation (construction),0910Q2,NaT,NaT
6,1659,30,ProtectedBikeLane,Pre-protected bike lane (minor change),1011Q4,2020-07-01,2020-10-01
7,1660,30,ProtectedBikeLane,Pre-protected bike lane (minor change),1011Q4,2020-07-01,2020-10-01
8,1661,30,ProtectedBikeLane,Pre-protected bike lane (minor change),1011Q4,2020-07-01,2020-10-01
9,1662,30,ProtectedBikeLane,Pre-protected bike lane (minor change),1011Q4,2013-04-01,2013-07-01


**Finding:** of the 87 street sgments with an identified intervention-completion quarter, 69 also have at least one recorded period where is_under_construction == 1, while 18 do not. the construction flags provide useful supporting context, although some recorded construction occurs outside the initial intervention period and will require more careful treatment if used in later before-and-after analysis. 

In [30]:
#--- 4.13 identify strongest sprint 1 candidate records ---#

#segments that have supporting sites_db information
segments_with_site_info = site_match_counts["StreetSegmentID"]

#keep segments with:
# 1. validated completion quarter
# 2. recorded construction info
# 3. supporting sites_db info
strong_candidates = sprint1_intervention_summary[
    sprint1_intervention_summary["construction_start"].notna()
    &
    sprint1_intervention_summary["STREET_SEGMENT_ID"].isin(
        segments_with_site_info
    )
].copy()

print(
    "strongest sprint 1 candidate segments:",
    len(strong_candidates)
)

display(
    strong_candidates[
        [
            "STREET_SEGMENT_ID",
            "InterventionType",
            "InterventionSummary",
            "completion_quarter",
            "construction_start",
            "construction_end"
        ]
    ].head(20)
)

strongest sprint 1 candidate segments: 69


,STREET_SEGMENT_ID,InterventionType,InterventionSummary,completion_quarter,construction_start,construction_end
0,123,ProtectedBikeLane,Protected bike lane,2122Q2,2021-10-01,2021-10-01
1,124,ProtectedBikeLane,Protected bike lane (open one direction),2122Q2,2021-10-01,2024-04-01
2,125,ProtectedBikeLane,Protected bike lane,2122Q2,2021-10-01,2021-10-01
3,858,ProtectedBikeLane,Protected bike lane,2021Q4,2021-01-01,2021-04-01
6,1659,ProtectedBikeLane,Pre-protected bike lane (minor change),1011Q4,2020-07-01,2020-10-01
7,1660,ProtectedBikeLane,Pre-protected bike lane (minor change),1011Q4,2020-07-01,2020-10-01
8,1661,ProtectedBikeLane,Pre-protected bike lane (minor change),1011Q4,2020-07-01,2020-10-01
9,1662,ProtectedBikeLane,Pre-protected bike lane (minor change),1011Q4,2013-04-01,2013-07-01
10,1663,ProtectedBikeLane,Pre-protected bike lane (minor change),1011Q4,2013-04-01,2013-07-01
11,1753,ProtectedBikeLane,Protected Bike Lane,1011Q4,2010-10-01,2011-04-01


**Finding:** Of the 87 street segments with an identified completion quarter, 69 also have recorded construction information and supporting sites_db records. These represent the most complete available records identified during Sprint 1 and provide the strongest starting point for later station matching. Some timing inconsistencies remain and should be considered during later before-and-after analysis.